# Psychology RAG — Development Notebook
RAG system over the OpenStax *Psychology 2e* PDF: chunking → embeddings → Chroma → retrieval → reranking → Gemini answer with citations.

## 1. Install dependencies

In [ ]:
!pip install -q langchain langchain-community langchain-google-genai langchain-text-splitters chromadb sentence-transformers pypdf

## 2. Download the PDF

In [ ]:
import requests

url = "https://assets.openstax.org/oscms-prodcms/media/documents/Psychology2e_WEB.pdf"

response = requests.get(url)

if response.status_code == 200:
    with open("psychology2e.pdf", "wb") as f:
        f.write(response.content)

    print("PDF downloaded successfully.")
else:
    print("Failed to download PDF.")
    print("Status code:", response.status_code)

## 3. Load pages and clean text

In [ ]:
import re
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("psychology2e.pdf")
pages = loader.load()

def clean_text(text):
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)
    # Remove leading/trailing spaces
    text = text.strip()
    return text

for page in pages:
    page.page_content = clean_text(page.page_content)

print("Text cleaning completed.")

## 4. Split into chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)
chunks = splitter.split_documents(pages)

for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i

print(f"Number of pages: {len(pages)}")
print(f"Number of chunks: {len(chunks)}")

## 5. Build the embeddings + Chroma vector store

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Load a small, fast embedding model (runs locally inside Colab, not on your device)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Build the vector store from the chunks
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="chroma_db"
)

print("Vector store created successfully")
print(f"Number of vectors stored: {vectorstore._collection.count()}")

## 6. Connect the Gemini LLM
The API key is entered securely at runtime (never hardcoded), so this notebook is safe to share or push to GitHub.

In [ ]:
import os
from getpass import getpass
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0.2,
    max_retries=4
)

print("LLM connected successfully")

## 7. Retriever

In [ ]:
# Turn the vector store into a retriever (returns top 10 most relevant chunks)
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

## 8. Reranker (cross-encoder)

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Reranker loaded successfully.")

## 9. Query preparation, reranking, dedup, context formatting

In [ ]:
def prepare_query(question):
    question = question.strip()

    if not question:
        raise ValueError("Question cannot be empty.")

    return question


def rerank_documents(question, documents, top_n=4, score_threshold=0.0):

    if not documents:
        return []

    pairs = [
        [question, doc.page_content]
        for doc in documents
    ]

    scores = reranker.predict(pairs)

    scored_documents = list(zip(documents, scores))

    # Sort from highest relevance score to lowest
    scored_documents.sort(
        key=lambda x: x[1],
        reverse=True
    )

    print("\nReranking Results:")
    print("-" * 60)

    for rank, (doc, score) in enumerate(scored_documents, start=1):
        page = doc.metadata.get("page", "Unknown")
        print(f"Rank {rank} | Score: {score:.4f} | Page: {page}")

    print("-" * 60)

    # Keep only documents above the threshold
    filtered_documents = [
        (doc, score)
        for doc, score in scored_documents
        if score >= score_threshold
    ]

    # Keep only top N
    final_documents = filtered_documents[:top_n]

    print(f"Documents after threshold: {len(filtered_documents)}")
    print(f"Final documents selected: {len(final_documents)}")

    return final_documents


def remove_duplicate_documents(scored_documents):
    unique_documents = []
    seen_content = set()

    for doc, score in scored_documents:
        content = doc.page_content.strip()

        if content not in seen_content:
            unique_documents.append((doc, score))
            seen_content.add(content)

    return unique_documents


def retrieve_and_rerank(question):
    question = prepare_query(question)

    retrieved_docs = retriever.invoke(question)
    print(f"\nRetrieved documents: {len(retrieved_docs)}")

    reranked_docs = rerank_documents(
        question,
        retrieved_docs,
        top_n=4,
        score_threshold=0.0
    )

    final_docs = remove_duplicate_documents(reranked_docs)

    return final_docs


def format_docs(scored_documents):
    formatted_docs = []

    for i, (doc, score) in enumerate(scored_documents, start=1):
        page = doc.metadata.get("page", "Unknown")
        source = doc.metadata.get("source", "Unknown")

        formatted_docs.append(
            f"""
[Source {i} | Page {page}]

Source file: {source}

{doc.page_content}
"""
        )

    return "\n\n".join(formatted_docs)


def get_context(question):
    docs = retrieve_and_rerank(question)
    return format_docs(docs)

## 10. Prompt + RAG chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""
You are an academic research assistant.

Answer the user's question using ONLY the information
provided in the context.

IMPORTANT CITATION RULES:

1. Use only the provided context.
2. Every important factual claim must have a citation.
3. Citations MUST use this exact format:
   [Source X | Page Y]
4. X and Y must come directly from the provided context.
5. Never invent a source number or page number.
6. If multiple sources support a claim, cite all relevant sources.
7. If the context does not contain enough information to answer
   the question, say:
   "I don't have enough information in the provided sources."
8. Do not use outside knowledge.

Context:
{context}

Question:
{question}

Answer:
""")

rag_chain = (
    {
        "context": get_context,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready")

## 11. Try it out

In [ ]:
question = "What is classical conditioning?"
answer = rag_chain.invoke(question)
print("\nFinal Answer:")
print(answer)

In [ ]:
question = "What is the current population of Egypt?"
answer = rag_chain.invoke(question)
print(answer)

## 12. Streamlit app file
Writes the deployable `app.py`. This is the same file included in the GitHub project (with the API key read from secrets, not hardcoded), reproduced here so the whole pipeline lives in one notebook too.

In [ ]:
%%writefile app.py

import os
import streamlit as st

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from sentence_transformers import CrossEncoder

from vectorstore_utils import build_or_load_vectorstore

st.set_page_config(page_title="Psychology RAG", page_icon="\U0001F4DA", layout="wide")

st.title("\U0001F4DA Psychology Book Q&A")
st.write("Ask questions about the Psychology 2e book using Retrieval-Augmented Generation.")

api_key = st.secrets.get("GOOGLE_API_KEY", os.environ.get("GOOGLE_API_KEY"))

if not api_key:
    st.error("GOOGLE_API_KEY is not set. Add it to Streamlit secrets or as an environment variable.")
    st.stop()

os.environ["GOOGLE_API_KEY"] = api_key


@st.cache_resource
def load_models():
    vectorstore = build_or_load_vectorstore()
    retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
    reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0.2, max_retries=4)
    return vectorstore, retriever, reranker, llm


vectorstore, retriever, reranker, llm = load_models()

# See vectorstore_utils.py and the GitHub repo's app.py for the full
# rerank / dedup / prompt / answer_question / UI code (identical logic
# to sections 9-11 above).

## 13. Quick local test with ngrok (optional, for use inside Colab only)
This is fine for a quick manual test while the notebook is open, but it is **not** a permanent deployment — the link dies when the Colab runtime stops. For an always-on app, deploy `app.py` on Streamlit Community Cloud instead (see the GitHub repo's README).

In [ ]:
!pip install -q streamlit pyngrok

In [ ]:
from pyngrok import ngrok
from getpass import getpass

# Enter tokens at runtime — never hardcode them
ngrok.set_auth_token(getpass("Enter your ngrok auth token: "))

get_ipython().system_raw("streamlit run app.py &>/content/logs.txt &")

public_url = ngrok.connect(8501)
print("Open your app here:", public_url)